In [2]:
import os
from dotenv import load_dotenv
load_dotenv() ## aloading all the environment variable

groq_api_key=os.getenv("GROQ_API_KEY")


In [3]:
from langchain_groq import ChatGroq
model=ChatGroq(model="openai/gpt-oss-120b",groq_api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x10e21f470>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10deda3f0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi , My name is Daksh and I am an AI Engineer")])

AIMessage(content='Hello Daksh! 👋 Nice to meet you. As an AI Engineer, you must be working on some exciting projects. How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says "Hi, My name is Daksh and I am an AI Engineer". Probably they are introducing themselves. The assistant should respond politely, maybe ask how can help. No policy issues. So respond friendly.'}, response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 84, 'total_tokens': 170, 'completion_time': 0.182350249, 'completion_tokens_details': {'reasoning_tokens': 45}, 'prompt_time': 0.027570096, 'prompt_tokens_details': None, 'queue_time': 0.615756943, 'total_time': 0.209920345}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_854fa9be4c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a05d97-7acb-71b3-bd47-9caa9eec66c3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8

In [5]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi , My name is Daksh and I am an AI Engineer"),
        AIMessage(content="Hello Daksh! It's nice to meet you. \n\nAs an AI Engineer, what kind of projects are you working on these days? \n\nI'm always eager to learn more about the exciting work being done in the field of AI.\n"),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]
)

AIMessage(content='Your name is **Daksh**, and you work as an **AI Engineer**.', additional_kwargs={'reasoning_content': 'The user asks: "Hey What\'s my name and what do I do?" We have prior conversation: user introduced themselves as Daksh, an AI Engineer. So answer: your name is Daksh and you are an AI Engineer. Should respond accordingly.'}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 150, 'total_tokens': 227, 'completion_time': 0.161360913, 'completion_tokens_details': {'reasoning_tokens': 51}, 'prompt_time': 0.006917473, 'prompt_tokens_details': None, 'queue_time': 1.481272892, 'total_time': 0.168278386}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c868cf1eaa', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a05d97-80b2-7da3-b5e1-7d6f590aa351-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 150, 'output_tokens': 77, 'total_tokens': 227, 'out

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [6]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

/var/folders/5y/_ykktgrx3hngv5rx63_dm2fr0000gn/T/ipykernel_1443/2274412368.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory
/opt/anaconda3/envs/tensorflow/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [7]:
config={"configurable":{"session_id":"chat1"}}

In [8]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Daksh and I am an AI Engineer")],
    config=config
)

In [9]:
response.content

'Hello Daksh! 👋 Great to meet a fellow AI Engineer. How can I assist you today?'

In [10]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , What is my name and what do I do?")],
    config=config
)

In [11]:
response.content

'Your name is **Daksh**, and you work as an **AI Engineer**. 🚀'

In [12]:
## Changing the session id to start a new conversation
config1 = {"configurable":{"session_id":"chat2"}}

response=with_message_history.invoke(
    [HumanMessage(content="Hi , What is my name and what do I do?")],
    config=config1
)

In [13]:
response.content

'I don’t have any information about you, so I’m not able to tell you your name or what you do. If you’d like to share a bit about yourself, I’ll be happy to continue the conversation!'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [14]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Amnswer all the question to the nest of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [15]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Daksh")]})

AIMessage(content='Hello, Daksh! Nice to meet you. How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "Hi My name is Daksh". Probably they want a greeting. Respond politely, ask how can help.'}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 98, 'total_tokens': 150, 'completion_time': 0.109548131, 'completion_tokens_details': {'reasoning_tokens': 26}, 'prompt_time': 0.004606574, 'prompt_tokens_details': None, 'queue_time': 1.723355192, 'total_time': 0.114154705}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_19b184c447', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a05d97-933c-7ae3-b727-aa2eb96e6ee8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 98, 'output_tokens': 52, 'total_tokens': 150, 'output_token_details': {'reasoning': 26}})

In [16]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

/opt/anaconda3/envs/tensorflow/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [17]:
config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Daksh")],
    config=config
)

response

AIMessage(content='Hello Daksh! 👋 Nice to meet you. How can I assist you today?', additional_kwargs={'reasoning_content': 'We need to respond. The user says "Hi My name is Daksh". Probably greeting. We can respond with greeting, ask how can help. Ensure friendly.'}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 98, 'total_tokens': 159, 'completion_time': 0.127597131, 'completion_tokens_details': {'reasoning_tokens': 34}, 'prompt_time': 0.010899678, 'prompt_tokens_details': None, 'queue_time': 0.396539283, 'total_time': 0.138496809}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_90620edd96', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a05d97-9b95-7e50-a5df-f5a29d9e1d25-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 98, 'output_tokens': 61, 'total_tokens': 159, 'output_token_details': {'reasoning': 34}})

In [18]:

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [19]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Daksh")],"language":"Hindi"})
response.content

'नमस्ते दक्ष! आपका स्वागत है। मैं आपकी कैसे मदद कर सकता हूँ?'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.


In [20]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

/opt/anaconda3/envs/tensorflow/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [21]:
config = {"configurable": {"session_id": "chat4"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Daksh")],"language":"Hindi"},
    config=config
)
repsonse.content

'नमस्ते, दक्ष! आपसे मिलकर खुशी हुई। आप कैसे हैं?'

In [22]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Hindi"},
    config=config,
)

response.content

'आपका नाम दक्ष है।'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
## 
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [23]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

/opt/anaconda3/envs/tensorflow/lib/python3.12/site-packages/langchain_core/language_models/base.py:463: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]